[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isrunej/Modul_Kinematik_Robot/blob/main/Modul_01_FK_2DOF.ipynb)

# Modul 1: Forward Kinematics — Robot 2-DOF Planar
## Studi Kasus: Robot Pemetik Stroberi di Greenhouse

---

### Tujuan Pembelajaran
Setelah menyelesaikan modul ini, mahasiswa mampu:
1. Menjelaskan konsep forward kinematics dengan kata-kata sendiri
2. Menghitung posisi end-effector dari sudut joint secara manual
3. Membuat simulasi visual robot 2-DOF dengan Python

---

### Konteks: Mengapa Ini Penting?

Di greenhouse modern, **robot arm** digunakan untuk memetik stroberi agar lebih efisien dan tidak merusak buah. Robot ini harus tahu **di mana posisi tangannya (end-effector)** setiap saat.

Bayangkan robot punya dua lengan yang terhubung:
- Lengan pertama (link 1) terhubung ke dudukan di rel greenhouse
- Lengan kedua (link 2) ujungnya adalah 'tangan' yang memetik

**Forward Kinematics menjawab pertanyaan:**
> *"Jika saya set motor sendi ke sudut tertentu, di mana posisi tangan robot?"*

```
  SUDUT JOINT (θ₁, θ₂)  →  [FORWARD KINEMATICS]  →  POSISI TANGAN (x, y)
```

---
## Bagian 1: Intuisi Visual

Sebelum matematika, mari kita bangun intuisi dulu.

Robot 2-DOF planar kita punya:
- **Link 1**: panjang $L_1$ (lengan atas)
- **Link 2**: panjang $L_2$ (lengan bawah / lengan pemetik)
- **Joint 1** ($\theta_1$): sudut dari sumbu horizontal ke Link 1
- **Joint 2** ($\theta_2$): sudut relatif Link 2 terhadap Link 1

![diagram](https://upload.wikimedia.org/wikipedia/commons/thumb/8/8a/2R_robot.svg/320px-2R_robot.svg.png)

### Rumus Posisi End-Effector

$$x = L_1 \cos(\theta_1) + L_2 \cos(\theta_1 + \theta_2)$$

$$y = L_1 \sin(\theta_1) + L_2 \sin(\theta_1 + \theta_2)$$

**Dari mana rumus ini?** Kita pakai **penjumlahan vektor**:
- Ujung Link 1 ada di $(L_1\cos\theta_1,\ L_1\sin\theta_1)$
- Dari ujung Link 1, Link 2 bergerak dengan sudut total $\theta_1 + \theta_2$

---
## Bagian 2: Implementasi Python — Langkah per Langkah

In [ ]:
# Import library yang kita butuhkan
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("Library berhasil diimport!")

In [ ]:
# ============================================
# PARAMETER ROBOT PEMETIK STROBERI
# ============================================

L1 = 0.4   # panjang lengan 1 = 40 cm
L2 = 0.3   # panjang lengan 2 = 30 cm

# Sudut joint dalam derajat
theta1_deg = 45   # sudut joint 1
theta2_deg = -30  # sudut joint 2 (negatif = menekuk ke bawah)

# Konversi ke radian (numpy pakai radian)
theta1 = np.radians(theta1_deg)
theta2 = np.radians(theta2_deg)

print(f"θ₁ = {theta1_deg}° = {theta1:.4f} rad")
print(f"θ₂ = {theta2_deg}° = {theta2:.4f} rad")

In [ ]:
# ============================================
# FUNGSI FORWARD KINEMATICS
# ============================================

def forward_kinematics_2dof(theta1, theta2, L1, L2):
    """
    Hitung posisi setiap joint dan end-effector robot 2-DOF planar.
    
    Input:
        theta1, theta2 : sudut joint dalam RADIAN
        L1, L2         : panjang link
    
    Output:
        dict berisi koordinat setiap titik
    """
    # Posisi base (pangkal robot) — tetap di origin
    x0, y0 = 0, 0
    
    # Posisi ujung Link 1 (= posisi Joint 2)
    x1 = L1 * np.cos(theta1)
    y1 = L1 * np.sin(theta1)
    
    # Posisi end-effector (ujung Link 2)
    x2 = x1 + L2 * np.cos(theta1 + theta2)
    y2 = y1 + L2 * np.sin(theta1 + theta2)
    
    return {
        'base':         (x0, y0),
        'joint2':       (x1, y1),
        'end_effector': (x2, y2)
    }

# Jalankan FK
hasil = forward_kinematics_2dof(theta1, theta2, L1, L2)

print("\n=== HASIL FORWARD KINEMATICS ===")
print(f"Posisi Base       : {hasil['base']}")
print(f"Posisi Joint 2    : ({hasil['joint2'][0]:.4f}, {hasil['joint2'][1]:.4f}) meter")
print(f"Posisi End-Effector: ({hasil['end_effector'][0]:.4f}, {hasil['end_effector'][1]:.4f}) meter")

---
## Bagian 3: Visualisasi Robot

In [ ]:
def visualisasi_robot_2dof(theta1_deg, theta2_deg, L1, L2, 
                            posisi_stroberi=None, judul='Robot Pemetik Stroberi'):
    """
    Visualisasikan robot 2-DOF dan (opsional) posisi stroberi.
    Input sudut dalam DERAJAT untuk kemudahan.
    """
    theta1 = np.radians(theta1_deg)
    theta2 = np.radians(theta2_deg)
    
    posisi = forward_kinematics_2dof(theta1, theta2, L1, L2)
    
    fig, ax = plt.subplots(1, 1, figsize=(8, 7))
    
    # --- Gambar Greenhouse (latar belakang) ---
    ax.fill_between([-0.1, 0.9], [-0.05, -0.05], [0, 0], 
                    color='#8B4513', alpha=0.4, label='Tanah')
    ax.fill_between([-0.1, 0.9], [0, 0], [-0.05, -0.05],
                    color='#8B4513', alpha=0.4)
    
    # --- Gambar Robot ---
    x0, y0 = posisi['base']
    x1, y1 = posisi['joint2']
    x2, y2 = posisi['end_effector']
    
    # Link 1
    ax.plot([x0, x1], [y0, y1], 'b-', linewidth=6, 
            solid_capstyle='round', label='Link 1 (lengan atas)', zorder=3)
    
    # Link 2  
    ax.plot([x1, x2], [y1, y2], 'g-', linewidth=5, 
            solid_capstyle='round', label='Link 2 (lengan bawah)', zorder=3)
    
    # Joint 1 (base)
    ax.plot(x0, y0, 'ko', markersize=16, zorder=5)
    ax.plot(x0, y0, 'wo', markersize=8, zorder=6)
    ax.annotate('Base\n(Joint 1)', (x0, y0), 
                textcoords='offset points', xytext=(-50, -25),
                fontsize=9, color='black')
    
    # Joint 2
    ax.plot(x1, y1, 'ko', markersize=14, zorder=5)
    ax.plot(x1, y1, 'wo', markersize=6, zorder=6)
    ax.annotate(f'Joint 2\n({x1:.2f}, {y1:.2f})', (x1, y1),
                textcoords='offset points', xytext=(10, 5),
                fontsize=9, color='black')
    
    # End-effector
    ax.plot(x2, y2, 'r*', markersize=18, zorder=5, label='End-Effector (tangan)')
    ax.annotate(f'End-Effector\n({x2:.2f}, {y2:.2f})', (x2, y2),
                textcoords='offset points', xytext=(10, -20),
                fontsize=9, color='darkred', fontweight='bold')
    
    # --- Gambar sudut ---
    import matplotlib.patches as mpatches
    arc1 = mpatches.Arc((x0, y0), 0.15, 0.15, angle=0, 
                         theta1=0, theta2=theta1_deg, color='blue', linewidth=2)
    ax.add_patch(arc1)
    ax.annotate(f'θ₁={theta1_deg}°', (x0+0.09, y0+0.03), 
                fontsize=10, color='blue', fontweight='bold')
    
    # --- Gambar stroberi (jika ada) ---
    if posisi_stroberi:
        for i, (sx, sy) in enumerate(posisi_stroberi):
            ax.plot(sx, sy, marker='*', color='red', markersize=20, zorder=4)
            ax.annotate(f' Stroberi {i+1}\n ({sx}, {sy})', (sx, sy),
                        fontsize=9, color='darkred')
    
    # --- Workspace circle (area jangkauan maksimal) ---
    workspace = plt.Circle((0, 0), L1+L2, fill=False, 
                            color='gray', linestyle='--', linewidth=1, alpha=0.5)
    workspace_min = plt.Circle((0, 0), abs(L1-L2), fill=False,
                               color='gray', linestyle=':', linewidth=1, alpha=0.5)
    ax.add_patch(workspace)
    ax.add_patch(workspace_min)
    
    # --- Sumbu referensi ---
    ax.axhline(y=0, color='k', linewidth=0.5, alpha=0.3)
    ax.axvline(x=0, color='k', linewidth=0.5, alpha=0.3)
    
    # --- Info kotak ---
    info_text = (f'Parameter Robot:\n'
                 f'  L₁ = {L1} m, L₂ = {L2} m\n'
                 f'  θ₁ = {theta1_deg}°, θ₂ = {theta2_deg}°\n'
                 f'\nHasil FK:\n'
                 f'  x = {x2:.4f} m\n'
                 f'  y = {y2:.4f} m')
    ax.text(0.02, 0.98, info_text, transform=ax.transAxes,
            verticalalignment='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    
    ax.set_xlim(-0.85, 0.85)
    ax.set_ylim(-0.2, 0.85)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('x (meter)', fontsize=11)
    ax.set_ylabel('y (meter)', fontsize=11)
    ax.set_title(judul + f'\nθ₁={theta1_deg}°, θ₂={theta2_deg}°', 
                 fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    return posisi

# Visualisasi pertama
posisi_stroberi = [(0.5, 0.3), (0.3, 0.5)]
hasil = visualisasi_robot_2dof(45, -30, L1, L2, posisi_stroberi)

---
## Bagian 4: Eksplorasi Interaktif — Coba Ubah Sudut!

In [ ]:
# ============================================================
# COBA EKSPLORASI: ubah nilai theta1 dan theta2 di bawah ini
# ============================================================

# Skenario 1: Robot menjangkau stroberi di kanan
visualisasi_robot_2dof(
    theta1_deg=20,
    theta2_deg=10,
    L1=L1, L2=L2,
    judul='Skenario 1: Menjangkau Stroberi di Kanan'
)

In [ ]:
# Skenario 2: Robot menjangkau stroberi di atas
visualisasi_robot_2dof(
    theta1_deg=70,
    theta2_deg=-20,
    L1=L1, L2=L2,
    judul='Skenario 2: Menjangkau Stroberi di Atas'
)

---
## Bagian 5: Transformasi Homogen (Penjelasan Matematis Lebih Dalam)

Cara lain yang lebih sistematis menggunakan **matriks transformasi homogen**.
Ini penting karena metode ini akan kita gunakan di robot 3D nanti.

Untuk rotasi 2D dan translasi, matriks $3 \times 3$ adalah:

$$T = \begin{bmatrix} \cos\theta & -\sin\theta & d_x \\ \sin\theta & \cos\theta & d_y \\ 0 & 0 & 1 \end{bmatrix}$$

Untuk robot 2-DOF:
$$T_{total} = T_1 \cdot T_2$$

In [ ]:
def matriks_transformasi_2d(theta, dx, dy):
    """Buat matriks transformasi homogen 2D (3x3)"""
    T = np.array([
        [np.cos(theta), -np.sin(theta), dx],
        [np.sin(theta),  np.cos(theta), dy],
        [0,              0,             1 ]
    ])
    return T

# Setup sudut
theta1_deg, theta2_deg = 45, -30
theta1 = np.radians(theta1_deg)
theta2 = np.radians(theta2_deg)

# T1: dari base ke ujung Link 1
# Rotasi theta1, translasi L1 ke arah sumbu lokal
T1 = matriks_transformasi_2d(theta1, L1, 0)

# T2: dari ujung Link 1 ke end-effector
T2 = matriks_transformasi_2d(theta2, L2, 0)

# Posisi end-effector relatif dari base
T_total = T1 @ T2  # operator @ = perkalian matriks

print("Matriks T1 (base → joint2):")
print(np.round(T1, 4))
print("\nMatriks T2 (joint2 → end-effector):")
print(np.round(T2, 4))
print("\nMatriks T_total (base → end-effector):")
print(np.round(T_total, 4))
print(f"\nPosisi End-Effector dari matriks: x = {T_total[0,2]:.4f}, y = {T_total[1,2]:.4f}")

# Verifikasi: harusnya sama dengan rumus langsung
x_check = L1*np.cos(theta1) + L2*np.cos(theta1+theta2)
y_check = L1*np.sin(theta1) + L2*np.sin(theta1+theta2)
print(f"\nVerifikasi rumus langsung:     x = {x_check:.4f}, y = {y_check:.4f}")
print("✅ Hasil sama!" if abs(T_total[0,2]-x_check) < 1e-10 else "❌ Ada perbedaan")

---
## Bagian 6: Visualisasi Workspace Robot

**Workspace** = semua titik yang bisa dijangkau robot.
Ini penting untuk merancang posisi robot di greenhouse!

In [ ]:
def plot_workspace(L1, L2, resolusi=50):
    """Plot semua posisi yang bisa dijangkau robot 2-DOF"""
    
    sudut_range = np.linspace(0, 2*np.pi, resolusi)
    
    x_semua = []
    y_semua = []
    
    for t1 in sudut_range:
        for t2 in sudut_range:
            x = L1*np.cos(t1) + L2*np.cos(t1+t2)
            y = L1*np.sin(t1) + L2*np.sin(t1+t2)
            x_semua.append(x)
            y_semua.append(y)
    
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(x_semua, y_semua, s=0.5, alpha=0.3, color='blue')
    
    # Tandai beberapa posisi stroberi contoh
    stroberi = [(0.5, 0.3), (0.3, 0.55), (-0.2, 0.6), (0.6, 0.1)]
    for sx, sy in stroberi:
        r = np.sqrt(sx**2 + sy**2)
        terjangkau = abs(L1-L2) <= r <= (L1+L2)
        warna = 'green' if terjangkau else 'red'
        label = 'Terjangkau' if terjangkau else 'Tidak terjangkau'
        ax.plot(sx, sy, '*', color=warna, markersize=18)
        ax.annotate(f'({sx},{sy})\n{label}', (sx, sy), 
                    textcoords='offset points', xytext=(5, 5), fontsize=8)
    
    ax.set_xlim(-0.85, 0.85)
    ax.set_ylim(-0.85, 0.85)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_title(f'Workspace Robot 2-DOF (L₁={L1}m, L₂={L2}m)\n'
                 f'Area biru = semua posisi yang bisa dijangkau', fontsize=12)
    ax.set_xlabel('x (meter)')
    ax.set_ylabel('y (meter)')
    
    plt.tight_layout()
    plt.show()

plot_workspace(L1, L2)

---
## Ringkasan Modul 1

| Konsep | Rumus |
|--------|-------|
| Posisi Joint 2 | $x_1 = L_1\cos\theta_1$, $y_1 = L_1\sin\theta_1$ |
| Posisi End-Effector | $x = x_1 + L_2\cos(\theta_1+\theta_2)$, $y = y_1 + L_2\sin(\theta_1+\theta_2)$ |
| Transformasi Homogen | $T_{total} = T_1 \cdot T_2$ |

**Poin kunci:**
- FK = "diberi sudut joint → cari posisi tangan"
- Sudut selalu dalam **radian** saat komputasi, tapi boleh input **derajat** lalu konversi
- Matriks transformasi homogen lebih sistematis untuk robot dengan banyak joint

---
## Lanjut ke Modul 2: Inverse Kinematics

Modul berikutnya: jika kita tahu **posisi stroberi** yang ingin dipetik, bagaimana menentukan **sudut jointnya?**